In [ ]:
from pathlib import Path
import importlib
import sys

# Resolve project root robustly whether notebook runs from project root or notebooks/
PROJECT_ROOT = (
    Path.cwd() if (Path.cwd() / "config" / "config.yaml").exists() else Path.cwd().parent
).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_module = importlib.import_module("data.load")
preprocess_module = importlib.import_module("data.preprocess")
health_index_module = importlib.import_module("features.health_index")
velocity_module = importlib.import_module("features.velocity")
variability_module = importlib.import_module("features.variability")
clustering_module = importlib.import_module("model.clustering")
risk_module = importlib.import_module("model.risk")
validation_module = importlib.import_module("evaluation.validation")
validation_module = importlib.reload(validation_module)

load_dataset = load_module.load_dataset
load_config = load_module.load_config
preprocess_train = preprocess_module.preprocess_train
preprocess_test = preprocess_module.preprocess_test
build_health_index = health_index_module.build_health_index
build_velocity = velocity_module.build_velocity
build_variability = variability_module.build_variability
build_clustering = clustering_module.build_clustering
build_risk_score = risk_module.build_risk_score
run_validation = validation_module.run_validation

config = load_config(PROJECT_ROOT / "config" / "config.yaml")
config["dataset"]["raw_path"] = str(PROJECT_ROOT / "data" / "raw")

train_raw, test_raw, _ = load_dataset(config)
train_proc, scaler, sensor_cols = preprocess_train(train_raw, config)
test_proc = preprocess_test(test_raw, config, scaler)
train_hi, test_hi, hi_art = build_health_index(train_proc, test_proc, config)
train_vel, test_vel, velocity_artifacts = build_velocity(train_hi, test_hi, config)
train_var, test_var, var_art = build_variability(train_vel, test_vel, config)
train_cl, test_cl, cl_art = build_clustering(train_var, test_var, config)
train_rs, test_rs, risk_art = build_risk_score(train_cl, test_cl, cl_art)

# Run validation
report = run_validation(train_rs, config=config)
report.print_report()

# Hard assertions
assert report.pct_monotonic_hi > 80.0, \
    f"Too many non-monotonic engines: {report.pct_monotonic_hi:.1f}%"
assert report.pct_valid_cluster > 70.0, \
    f"Too many invalid cluster progressions: {report.pct_valid_cluster:.1f}%"
assert report.risk_rul_pearson_r < report.weak_risk_rul_correlation_threshold, \
    f"Risk-RUL correlation too weak: {report.risk_rul_pearson_r:.4f}"

print("\n✅ All validation assertions passed")

[load] Train shape : (20631, 26)
[load] Test shape  : (13096, 26)
[load] RUL entries : 100
PIPELINE VALIDATION REPORT

[1] Health Index Monotonicity
    Engines validated: 100
    Mean Spearman rho: -0.9250
    Monotonic (|rho| >= 0.7): 100.0%

[2] Cluster Progression Consistency
    Valid progression: 0.0%

[3] Risk Score-RUL Correlation
    Pearson r: -0.7683 (p-value: 0.00e+00)
    Interpretation: Strong negative correlation - risk tracks degradation correctly.

[!] Anomalous engines detected (flagged for review)
    Unit IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100]


AssertionError: Too many invalid cluster progressions: 0.0%